# CTDenoiser — Training Notebook

| Part | What happens |
|------|-------------|
| **0 — Setup** | Mount Drive, clone/update repo, install deps |
| **1 — Smoke test** | Verify everything works on synthetic data |
| **2 — Train** | Train RED-CNN and CTformer on real data |
| **3 — W&B sweep** | Bayesian HP search over both architectures |

**Prerequisites:** Run `00_build_cache.ipynb` once to download the TCIA LDCT data and build `ldct_cache.h5` on your Drive.

---
## Part 0 — Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL  = 'https://github.com/tsereda/CTDenoiser.git'
BRANCH    = 'main'
REPO_ROOT = '/content/drive/MyDrive/CTDenoiser'

if not os.path.isdir(os.path.join(REPO_ROOT, '.git')):
    os.makedirs(REPO_ROOT, exist_ok=True)
    !git clone --branch {BRANCH} {REPO_URL} {REPO_ROOT}
    print('Cloned.')
else:
    print('Repo already exists, pulling latest.')

%cd {REPO_ROOT}
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}

In [ ]:
%cd {REPO_ROOT}
!pip install -q -r requirements.txt

In [ ]:
%cd {REPO_ROOT}
!pytest -q

---
## Part 1 — Smoke Test (Synthetic Data)

Verifies the training loop runs end-to-end without needing real CT data.

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train --model redcnn --epochs 1 --synthetic-len 32
print('RED-CNN smoke test passed.')

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train --model ctformer --epochs 1 --synthetic-len 32
print('CTformer smoke test passed.')

---
## Part 2 — Train RED-CNN and CTformer on Real Data

Copies the HDF5 cache to fast local disk, then trains both models sequentially.
Adjust `--epochs` and `--batch-size` to your GPU budget.

In [ ]:
import os, shutil

DRIVE_CACHE = '/content/drive/MyDrive/ldct_cache.h5'
LOCAL_CACHE = '/content/ldct_cache.h5'

if os.path.exists(DRIVE_CACHE) and not os.path.exists(LOCAL_CACHE):
    print('Copying cache to local disk for faster I/O...')
    shutil.copy(DRIVE_CACHE, LOCAL_CACHE)
    print('Done.')

assert os.path.exists(LOCAL_CACHE), (
    'ldct_cache.h5 not found — run 00_build_cache.ipynb first, '
    'or place the file at ' + DRIVE_CACHE
)

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train \
    --model redcnn \
    --h5-cache /content/ldct_cache.h5 \
    --epochs 1 \
    --batch-size 16 \
    --patch-size 64 \
    --lr 1e-4

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train \
    --model ctformer \
    --h5-cache /content/ldct_cache.h5 \
    --epochs 1 \
    --batch-size 16 \
    --patch-size 64 \
    --lr 1e-4

---
## Part 3 — Weights & Biases Hyperparameter Sweep

Bayesian search over learning rate, batch size, patch size, **and model architecture**
(RED-CNN vs CTformer). Each run is logged to your W&B project so you can compare
them side-by-side.

**Before running:** log in to W&B once with `wandb login` or set `WANDB_API_KEY`.

In [ ]:
import wandb
wandb.login()   # prompts for API key if not already set

In [ ]:
SWEEP_PROJECT = 'ctdenoiser'

# How many total runs to launch (increase for a thorough search)
SWEEP_COUNT = 20

# Uses the same local cache copied in Part 2 above
SWEEP_H5_CACHE = LOCAL_CACHE

sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'val/psnr', 'goal': 'maximize'},
    'parameters': {
        'model':      {'values': ['redcnn', 'ctformer']},
        'lr':         {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 1e-3},
        'batch_size': {'values': [8, 16, 32]},
        'patch_size': {'values': [64, 96]},
        'epochs':     {'value': 5},
    },
}

sweep_id = wandb.sweep(sweep_config, project=SWEEP_PROJECT)
print(f'Sweep created: {sweep_id}')

In [ ]:
import subprocess, sys, os

def train_run():
    """Single W&B sweep run: delegates to train.py so no training logic lives here."""
    run = wandb.init()
    cfg = run.config

    cmd = [
        sys.executable, '-m', 'ctdenoiser.train',
        '--model',       cfg.model,
        '--lr',          str(cfg.lr),
        '--batch-size',  str(cfg.batch_size),
        '--patch-size',  str(cfg.patch_size),
        '--epochs',      str(cfg.epochs),
        '--wandb-project', SWEEP_PROJECT,
    ]
    if SWEEP_H5_CACHE and os.path.exists(SWEEP_H5_CACHE):
        cmd += ['--h5-cache', SWEEP_H5_CACHE]

    # Pass run ID so train.py resumes this run and logs per-epoch metrics.
    env = {**os.environ, 'WANDB_RUN_ID': run.id, 'WANDB_RESUME': 'must'}
    subprocess.run(cmd, env=env, check=True, cwd=REPO_ROOT)

wandb.agent(sweep_id, function=train_run, count=SWEEP_COUNT)

### Viewing Results

Open [wandb.ai](https://wandb.ai) → your project `ctdenoiser` → **Sweeps** tab.

Useful views:
- **Parallel coordinates** — see which HP combos dominate on val/psnr
- **Scatter plot** `model` vs `val/psnr` — direct RED-CNN vs CTformer comparison
- **Importance** panel — which hyperparameter matters most